In [15]:
import torch
from torch.utils.data import Dataset,DataLoader

class GPTDatasetv1(Dataset):
    def __init__(self,text,max_length,tokenizer,stride):
        self.inputs=[]
        self.targets=[]

        tiktoken=tokenizer.encode(text,allowed_special={"<|endoftext|>"})
        for i in range(0,len(tiktoken)-max_length,stride):
            input_seq=tiktoken[i:i+max_length]
            target_seq=tiktoken[i+1:i+max_length+1]

            self.inputs.append(torch.tensor(input_seq))
            self.targets.append(torch.tensor(target_seq))
        
    def __len__(self):
            return len(self.inputs)
        
    def __getitem__(self,idx):
            return self.inputs[idx],self.targets[idx]
        

In [6]:
import os
os.getcwd()

'c:\\Users\\panka\\Desktop\\LLM\\LLM_from_Scratch\\11.The importance of positional Embedding'

In [7]:
with open(r"c:\\Users\\panka\\Desktop\\LLM\\LLM_from_Scratch\\the-verdict.txt","r",encoding="utf-8") as f:
    text=f.read()

In [16]:
def create_dataloader_v1(text,batch_size=4,max_length=256,stride=128,shuffle=True,num_workers=0,drop_last=True):
    import tiktoken
    tokenizer=tiktoken.get_encoding("gpt2")

    dataset=GPTDatasetv1(text,max_length,tokenizer,stride)

    dataloader=DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        drop_last=drop_last
    )

    return dataloader
    

In [17]:
dataloader = create_dataloader_v1(
    text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [18]:
secod_batch = next(data_iter)
print(secod_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [19]:
dataloader = create_dataloader_v1(text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [21]:
vocab_size=50257
output_dim=256
torch.manual_seed(123)

embedding_layer=torch.nn.Embedding(vocab_size,output_dim)
print("Embedding weights:",embedding_layer.weight.shape)
print("Embedding weights:",embedding_layer.weight)

Embedding weights: torch.Size([50257, 256])
Embedding weights: Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035,  ...,  1.3337,  0.0771, -0.0522],
        [ 0.2386,  0.1411, -1.3354,  ..., -0.0315, -1.0640,  0.9417],
        [-1.3152, -0.0677, -0.1350,  ..., -0.3181, -1.3936,  0.5226],
        ...,
        [ 0.5871, -0.0572, -1.1628,  ..., -0.6887, -0.7364,  0.4479],
        [ 0.4438,  0.7411,  1.1263,  ...,  1.2091,  0.6781,  0.3331],
        [-0.2537,  0.1446,  0.7203,  ..., -0.2134,  0.2144,  0.3006]],
       requires_grad=True)


In [24]:
max_length=4
dataloader=create_dataloader_v1(
    text,batch_size=8,max_length=max_length,stride=max_length,shuffle=False
)

data_iter=iter(dataloader)
inputs,targets=next(data_iter)
print("Input IDs:\n",inputs)

Input IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])


In [25]:
print("\nInputs shape:\n", inputs.shape)


Inputs shape:
 torch.Size([8, 4])


In [26]:
token_embeddings = embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [27]:
#each token ID is now embedded as a 256-dimensional vector.

In [29]:
context_length=max_length
pos_embedding_layer=torch.nn.Embedding(context_length,output_dim)

In [30]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [31]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
